## Import Libraries

In [3]:
import arxiv
import json
import os
from typing import List
from dotenv import load_dotenv
import anthropic

## Tool Functions

In [4]:
PAPER_DIR = "papers"

### First Tool

In [5]:
def search_papers(topic: str, max_results: int = 5) -> List[str]:
    """
    Search for papers on arXiv based on a topic and store their information.
    
    Args:
        topic: The topic to search for
        max_results: Maximum number of results to retrieve (default: 5)
        
    Returns:
        List of paper IDs found in the search
    """
    
    # Use arxiv to find the papers 
    client = arxiv.Client()

    # Search for the most relevant articles matching the queried topic
    search = arxiv.Search(
        query = topic,
        max_results = max_results,
        sort_by = arxiv.SortCriterion.Relevance
    )

    papers = client.results(search)
    
    # Create directory for this topic
    path = os.path.join(PAPER_DIR, topic.lower().replace(" ", "_"))
    os.makedirs(path, exist_ok=True)
    
    file_path = os.path.join(path, "papers_info.json")

    # Try to load existing papers info
    try:
        with open(file_path, "r") as json_file:
            papers_info = json.load(json_file)
    except (FileNotFoundError, json.JSONDecodeError):
        papers_info = {}

    # Process each paper and add to papers_info  
    paper_ids = []
    for paper in papers:
        paper_ids.append(paper.get_short_id())
        paper_info = {
            'title': paper.title,
            'authors': [author.name for author in paper.authors],
            'summary': paper.summary,
            'pdf_url': paper.pdf_url,
            'published': str(paper.published.date())
        }
        papers_info[paper.get_short_id()] = paper_info
    
    # Save updated papers_info to json file
    with open(file_path, "w") as json_file:
        json.dump(papers_info, json_file, indent=2)
    
    print(f"Results are saved in: {file_path}")
    
    return paper_ids

### Check the function

In [8]:
search_papers("physics", 4)

Results are saved in: papers/physics/papers_info.json


['1209.0592v1', 'math-ph/0002018v2', 'math-ph/0503039v1', 'physics/0308107v1']

### Second Tool

In [9]:
def extract_info(paper_id: str) -> str:
    """
    Search for information about a specific paper across all topic directories.
    
    Args:
        paper_id: The ID of the paper to look for
        
    Returns:
        JSON string with paper information if found, error message if not found
    """
 
    for item in os.listdir(PAPER_DIR):
        item_path = os.path.join(PAPER_DIR, item)
        if os.path.isdir(item_path):
            file_path = os.path.join(item_path, "papers_info.json")
            if os.path.isfile(file_path):
                try:
                    with open(file_path, "r") as json_file:
                        papers_info = json.load(json_file)
                        if paper_id in papers_info:
                            return json.dumps(papers_info[paper_id], indent=2)
                except (FileNotFoundError, json.JSONDecodeError) as e:
                    print(f"Error reading {file_path}: {str(e)}")
                    continue
    
    return f"There's no saved information related to paper {paper_id}."

In [10]:
extract_info("physics/0308107v1")

'{\n  "title": "Contents of Physics Related E-Print Archives",\n  "authors": [\n    "E. R. Prakasan",\n    "Anil Kumar",\n    "Anil Sagar",\n    "Lalit Mohan",\n    "Sanjay Kumar Singh",\n    "V. L. Kalyane",\n    "Vijai Kumar"\n  ],\n  "summary": "The frontiers of physics related e-print archives (1994-2002) at\\nhttp://www.arxiv.org/archives/physics web service are explored from 7770\\nsubmissions. No. of e-prints in the six research disciplines besides physics\\n(5390) were: Condensed matter(754), Quantum physics(279), Astrophysics(222),\\nChemical physics(129), High energy physics Phenomenology(118), and High energy\\nphysics-Theory(100)). By keyword contents following major sub-fields have high\\nfrequency: Atomic physics(1258), General physics(1121), Chemical physics(892),\\nAccelerator physics(769), Optics(686), Biological physics(674), and\\nComputational physics(607). Interdomainary co-word cluster analysis revealed\\nhigher e-print contents for: Classical physics-General phys

## Tool Schema

In [12]:
tools = [
    {
        "name": "search_papers",
        "description": "Search for papers on arXiv based on a topic and store their information.",
        "input_schema": {
            "type": "object",
            "properties": {
                "topic": {
                    "type": "string",
                    "description": "The topic to search for"
                }, 
                "max_results": {
                    "type": "integer",
                    "description": "Maximum number of results to retrieve",
                    "default": 5
                }
            },
            "required": ["topic"]
        }
    },
    {
        "name": "extract_info",
        "description": "Search for information about a specific paper across all topic directories.",
        "input_schema": {
            "type": "object",
            "properties": {
                "paper_id": {
                    "type": "string",
                    "description": "The ID of the paper to look for"
                }
            },
            "required": ["paper_id"]
        }
    }
]

## Tool Mapping

In [14]:
mapping_tool_function = {
    "search_papers": search_papers,
    "extract_info": extract_info
}

def execute_tool(tool_name, tool_args):
    
    result = mapping_tool_function[tool_name](**tool_args)

    if result is None:
        result = "The operation completed but didn't return any results."
        
    elif isinstance(result, list):
        result = ', '.join(result)
        
    elif isinstance(result, dict):
        # Convert dictionaries to formatted JSON strings
        result = json.dumps(result, indent=2)
    
    else:
        # For any other type, convert using str()
        result = str(result)
    return result

## Chatbot Code

In [24]:
load_dotenv() 
client = anthropic.Anthropic()

In [25]:
def process_query(query):
    
    messages = [{'role': 'user', 'content': query}]
    
    response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages)
    
    process_query = True
    while process_query:
        assistant_content = []

        for content in response.content:
            if content.type == 'text':
                
                print(content.text)
                assistant_content.append(content)
                
                if len(response.content) == 1:
                    process_query = False
            
            elif content.type == 'tool_use':
                
                assistant_content.append(content)
                messages.append({'role': 'assistant', 'content': assistant_content})
                
                tool_id = content.id
                tool_args = content.input
                tool_name = content.name
                print(f"Calling tool {tool_name} with args {tool_args}")
                
                result = execute_tool(tool_name, tool_args)
                messages.append({"role": "user", 
                                  "content": [
                                      {
                                          "type": "tool_result",
                                          "tool_use_id": tool_id,
                                          "content": result
                                      }
                                  ]
                                })
                response = client.messages.create(max_tokens = 2024,
                                  model = 'claude-3-7-sonnet-20250219', 
                                  tools = tools,
                                  messages = messages) 
                
                if len(response.content) == 1 and response.content[0].type == "text":
                    print(response.content[0].text)
                    process_query = False

## Chat Loop

In [26]:
def chat_loop():
    print("Type your queries or 'quit' to exit.")
    while True:
        try:
            query = input("\nQuery: ").strip()
            if query.lower() == 'quit':
                break
    
            process_query(query)
            print("\n")
        except Exception as e:
            print(f"\nError: {str(e)}")

In [28]:
chat_loop()

Type your queries or 'quit' to exit.



Query:  I want to find 2 papers about RAG


I'll help you find 2 papers about RAG (Retrieval-Augmented Generation). Let me search for those papers on arXiv for you.
Calling tool search_papers with args {'topic': 'RAG Retrieval-Augmented Generation', 'max_results': 2}
Results are saved in: papers/rag_retrieval-augmented_generation/papers_info.json
Now, let me get more information about these two papers:
Calling tool extract_info with args {'paper_id': '2407.21059v1'}
Calling tool extract_info with args {'paper_id': '2501.00353v1'}
Here are the two papers about RAG (Retrieval-Augmented Generation) I found for you:

1. **Modular RAG: Transforming RAG Systems into LEGO-like Reconfigurable Frameworks**
   - **Authors**: Yunfan Gao, Yun Xiong, Meng Wang, Haofen Wang
   - **Published**: July 26, 2024
   - **Summary**: This paper introduces a modular RAG framework that decomposes complex RAG systems into independent modules and specialized operators. It moves beyond the traditional linear "retrieve-then-generate" architecture to a more 


Query:  I interesting in this article 2407.21059v1


I'll help you find information about the paper with the ID you've provided. Let me look that up for you.
Calling tool extract_info with args {'paper_id': '2407.21059v1'}
## Information about the Paper

I've found the paper you're interested in. Here are the details:

### "Modular RAG: Transforming RAG Systems into LEGO-like Reconfigurable Frameworks"

**Authors:**
- Yunfan Gao
- Yun Xiong
- Meng Wang
- Haofen Wang

**Published:** July 26, 2024

**arXiv ID:** 2407.21059v1

**Summary:**
This paper focuses on enhancing Retrieval-augmented Generation (RAG) systems. The authors note that as RAG applications have become more complex, the traditional "retrieve-then-generate" paradigm has become limiting. In response, they introduce a "modular RAG framework" that breaks down complex RAG systems into independent modules and specialized operators, creating a highly reconfigurable framework.

The framework moves beyond linear architecture to incorporate routing, scheduling, and fusion mechanisms.


Query:  quit
